# 14 · Modelos con y sin régimen: walk-forward

En los notebooks 11–13 examinamos modelos con CPCV. Ahora construiremos una referencia que respete el orden cronológico: **aprender del pasado y predecir el año siguiente**. Primero construiremos la referencia y después mediremos qué aporta añadir información del entorno.

- **M0:** frecuencia de éxitos del entrenamiento de cada año.
- **M1:** logística L2 con retorno del último mes y momentum 12–1.
- **M2:** la misma logística, añadiendo volatilidad, retorno de 24 meses y drawdown de 12 meses.
- **M3:** M2 más la interacción mercado bajista × volatilidad.

«Sin régimen» es una definición operativa: estas dos variables describen las señales y también pueden reflejar indirectamente el mercado. M2 añade las tres variables directas de régimen. M3 estudia la interacción previamente prevista, la última parte estudia también las tres retiradas individuales de M2.

**Entradas:** tablas de 01 → 02 → 07 → 08 y el entorno `.venv` con el extra `models`. No necesitas repetir los notebooks 09–13. El diseño, motivado por la guía del 15 de noviembre, está en [el protocolo](../docs/validacion_walk_forward.md). Se ha definido después de conocer los experimentos anteriores.

## 1. Un calendario que avanza año a año

El desarrollo sigue siendo 1927–1989. Para disponer de historia inicial, empezamos las pruebas en 1950. Aprendemos con la historia anterior y mantenemos los coeficientes fijos durante los doce meses de ese año. Al año siguiente ampliamos el entrenamiento y volvemos a ajustar. Son cuarenta periodos anuales de prueba, hasta 1989.

Aplicamos una purga de un mes objetivo anterior. Para probar 1950 usamos etiquetas hasta noviembre de 1949, diciembre de 1949 queda excluido. El calendario se define antes de retirar meses sin señal o sin características. Los meses futuros nunca entrenan ese ajuste. Como no hay entrenamiento posterior a su prueba, el embargo posterior de CPCV no elimina meses aquí.

La normalización causal de las características sí se actualiza con la historia anterior a cada nueva observación. Mantener los coeficientes durante el año no implica congelar en enero esa preparación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, save_result
from tfm.walk_forward import (FIRST_TEST_YEAR, LAST_TEST_YEAR, SIGNAL_COLUMNS,
    build_walk_forward_splits, load_walk_forward_inputs, summarize_walk_forward, run_walk_forward)

ROOT = project_root()
ejemplos, _, _ = load_walk_forward_inputs(ROOT)
particiones = build_walk_forward_splits()
cobertura = summarize_walk_forward(ejemplos, particiones)
display(cobertura.loc[cobertura.test_year.isin([FIRST_TEST_YEAR, LAST_TEST_YEAR])])

colores = {'train': ('Entrenamiento', '#9dc8b4'), 'purge': ('Purga', '#b74e40'),
           'validation': ('Prueba del año', '#244b70'), 'future': ('Futuro excluido', '#eeeeee')}
matriz = particiones.pivot(index='test_year', columns='target_period', values='role')
codigos = {nombre: i for i, nombre in enumerate(colores)}
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(matriz.replace(codigos).to_numpy(dtype=int), aspect='auto', interpolation='nearest',
          cmap=ListedColormap([color for _, color in colores.values()]), vmin=-.5, vmax=3.5)
x = [i for i, mes in enumerate(matriz.columns) if mes.endswith('-01') and int(mes[:4]) % 10 == 0]
y = list(range(0, len(matriz), 5))
ax.set_xticks(x, [matriz.columns[i][:4] for i in x])
ax.set_yticks(y, [matriz.index[i] for i in y])
ax.set(title='Historia disponible para cada año de prueba', xlabel='Mes objetivo del calendario', ylabel='Año de prueba')
fig.legend(handles=[Patch(color=color, label=nombre) for nombre, color in colores.values()],
           loc='lower center', ncol=4, fontsize=9)
fig.tight_layout(rect=(0, .06, 1, 1))
plt.show()

## 2. Qué ejemplos usamos y qué aprende M1

Una etiqueta indica si la operación aislada de la regla, neta de costes y dividendos aproximados, superó al efectivo. Solo tienen etiqueta las señales activas. Un mes sin operación no es un fallo del filtro y no recibe probabilidad cero.

Conservamos la muestra común de los experimentos anteriores: las diez características preparadas deben estar disponibles tras su calentamiento. M1 solo recibe **dos columnas explícitas**: `z_price_return_1m` y `z_momentum_12_1`. Las restantes no se utilizan para predecir. Así las comparaciones posteriores tendrán las mismas fechas, en lugar de ampliar la muestra de cada modelo por separado.

Usamos la misma logística regularizada del 11, con C=1, sin ponderar clases ni buscar parámetros. Las dos reglas se ajustan por separado. Los años sin señales permanecen en la cobertura. Si solo hay una clase de prueba, la log-loss se puede calcular y el AUC no.

In [ ]:
print('Predictores de M1:', SIGNAL_COLUMNS)
display(cobertura.groupby('strategy').agg(
    ejemplos_iniciales=('train_examples', 'min'),
    ejemplos_prueba=('validation_examples', 'sum'),
    minimo_anual=('validation_examples', 'min'),
    maximo_anual=('validation_examples', 'max')))
display(cobertura.loc[cobertura.validation_classes.lt(2),
    ['test_year', 'strategy', 'validation_calendar_months', 'no_signal_months',
     'excluded_active_months', 'validation_examples', 'validation_classes']])

## 3. Ajustar con el pasado y predecir una sola vez

M0 asigna a cada ejemplo del año la proporción de etiquetas positivas de su entrenamiento. M1 estima una probabilidad a partir de las dos características. Ninguno utiliza las etiquetas del año probado para ajustarse.

A diferencia del resumen de CPCV, cada ejemplo recibe **una única predicción cronológica** de cada modelo. No se promedian cinco modelos ni se cuentan repetidamente los mismos resultados. Si un ajuste no es estimable, sus probabilidades quedan ausentes, no se eliminan esos meses para mejorar el resumen.

In [ ]:
def mostrar_progreso(completadas, total):
    if completadas == 1 or completadas % 5 == 0:
        print(f'Años evaluados: {completadas}/{total}', flush=True)

tablas, informe = run_walk_forward(ROOT, progress=mostrar_progreso)
resumen = tablas['walk_forward_summary']
predicciones = tablas['walk_forward_predictions']
metricas = tablas['walk_forward_metrics']
coeficientes = tablas['walk_forward_coefficients']
print('Estado del recorrido:', informe['experiment_status'])
display(resumen[['test_year', 'strategy', 'train_first_target', 'train_last_target',
                'train_examples', 'validation_examples', 'train_classes', 'validation_classes', 'status']])

## 4. Leer los resultados

**AUC** describe si las etiquetas favorables reciben probabilidades más altas que las desfavorables. **Log-loss** penaliza probabilidades equivocadas, especialmente las muy seguras. Menor log-loss es favorable, mayor AUC indica mejor ordenación. Ninguna mide por sí sola rentabilidad.

El resumen principal (`pooled`) reúne las predicciones de prueba de los cuarenta años por regla y calcula la métrica una sola vez. No es una media de las AUC anuales. M0 tiene una probabilidad constante dentro de cada año, pero puede cambiar al reentrenarse: su AUC reunida puede diferir de 0,5.

Mostramos también cada año. `single_class` significa que no podemos calcular AUC porque todas las etiquetas son iguales, `no_examples`, que no hubo ejemplos. Estos años permanecen en las tablas. Si faltase una predicción necesaria, el resumen correspondiente quedaría no disponible.

In [ ]:
display(metricas.loc[metricas.aggregation.eq('pooled'),
    ['strategy', 'model', 'examples', 'positive', 'negative', 'log_loss', 'auc', 'status', 'auc_status']])
anuales = metricas.loc[metricas.aggregation.eq('year')]
display(anuales[['test_year', 'strategy', 'model', 'examples', 'log_loss', 'auc', 'status', 'auc_status']])
display(anuales.loc[anuales.model.eq('M1')].groupby(['strategy', 'auc_status']).size().rename('Años').reset_index())

## 5. Ver las probabilidades a lo largo del tiempo

Las líneas muestran las estimaciones de M0 y M1. Los puntos permiten ver también los meses aislados. Los huecos son meses sin ejemplo utilizable: no los rellenamos ni conectamos mediante interpolación. Una probabilidad alta significa que el modelo considera más probable superar al efectivo según nuestra etiqueta, no representa el tamaño de una ganancia ni una recomendación de inversión.

In [ ]:
import matplotlib.dates as mdates

meses_prueba = pd.period_range(f'{FIRST_TEST_YEAR}-01', f'{LAST_TEST_YEAR}-12', freq='M').astype(str)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    serie = predicciones.loc[predicciones.strategy.eq(regla)].set_index('target_period').reindex(meses_prueba)
    fechas = pd.PeriodIndex(serie.index, freq='M').to_timestamp()
    ax.plot(fechas, serie.probability_naive, label='M0 · frecuencia pasada', color='#c77832', linewidth=1.2, marker='.', markersize=2)
    ax.plot(fechas, serie.probability_logistic, label='M1 · información de las señales', color='#244b70', linewidth=1, marker='.', markersize=2)
    ax.set(title=regla, ylabel='Probabilidad estimada', ylim=(0, 1))
    ax.grid(alpha=.2)
axes[0].legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Mes objetivo de prueba')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
plt.show()

## 6. Guardar la referencia M0/M1

Esta primera parte establece y guarda la referencia para estudiar qué aporta el entorno. La sección siguiente añade M2 con las mismas fechas, ejemplos y parámetros. No seleccionamos variables ni umbrales de operación. Las sombras de la logística de diez características no validan automáticamente estas variantes.

El orden temporal evita entrenar con resultados posteriores al año de prueba. Sin embargo, seguimos usando precios medios mensuales de Shiller, dividendos interpolados y versiones actuales de las fuentes. La purga no certifica sus fechas históricas de publicación. Es una aproximación retrospectiva, no acredita operaciones ejecutables ni disponibilidad point-in-time. El periodo desde 1990 permanece reservado.

La siguiente celda guarda las cinco tablas de referencia con sus JSON. Es necesario publicarlas antes de ejecutar M2. Repetir su publicación retira las comparaciones de régimen anteriores, que deberán recalcularse. El historial del intento está en `outputs/experiments/walk_forward_trials.jsonl`. Si falla el guardado, repite la celda completa. Antes del commit, limpia las salidas y guarda.

Referencia: guía del tutor del 15 de noviembre de 2026, apartado 5, pp. 19–21. La adaptación, el calendario y los supuestos están en [el protocolo](../docs/validacion_walk_forward.md). Se reutilizan las metodologías y referencias de preparación, etiquetado y logística ya documentadas en el proyecto.

In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados))
print('Historial:', informe['trial_log'])
print('Evaluación pendiente:', informe['skill_gate_status'])

## 7. Qué información añade M2

Ahora planteamos una comparación concreta: ¿la logística mejora al conocer también las condiciones del mercado? M2 conserva las dos entradas de M1 y añade tres variables ya construidas en el notebook 07:

- **Volatilidad de 12 meses:** cuánto han variado los retornos recientes.
- **Retorno de 24 meses:** si el mercado venía subiendo o cayendo y con qué magnitud.
- **Drawdown de 12 meses:** distancia del precio al máximo de esa ventana.

Puede verse como comparar dos decisiones con el mismo historial de señales: una dispone de esa información básica y la otra conoce también el entorno. Todo lo demás permanece igual: logística L2 C=1, ejemplos, etiquetas, años, purga y transformación causal.

Las cinco columnas están fijadas por el protocolo. No se escogen según el resultado de M2, no se usa todavía Clustered MDA y la interacción mercado bajista × volatilidad se estudia por separado en la sección de M3. Esta comparación mide la aportación del **conjunto de tres variables**, no demuestra qué parte corresponde a cada una.

In [ ]:
from tfm.walk_forward import REGIME_COLUMNS, M2_COLUMNS, run_walk_forward_regime

print('M1 · información de las señales:', SIGNAL_COLUMNS)
print('M2 · variables de régimen añadidas:', REGIME_COLUMNS)
print('Total de predictores de M2:', len(M2_COLUMNS))

tablas_regimen, informe_regimen = run_walk_forward_regime(ROOT, progress=mostrar_progreso)
comparacion_regimen = tablas_regimen['walk_forward_regime_comparison']
metricas_regimen = tablas_regimen['walk_forward_regime_metrics']
predicciones_regimen = tablas_regimen['walk_forward_regime_predictions']
print('Estado del recorrido M2:', informe_regimen['experiment_status'])
display(pd.DataFrame(informe_regimen['diagnostics'])[
    ['test_year', 'strategy', 'train_examples', 'validation_examples', 'train_positive', 'iterations', 'status']])

## 8. Comparar los mismos ejemplos

La tabla calcula **AUC de M2 menos AUC de M1**. Una diferencia positiva indica que añadir el régimen mejora la ordenación de las etiquetas. También calcula **log-loss de M1 menos log-loss de M2**, para que una cifra positiva indique mejora en ambos casos. Son diferencias de métricas, no porcentajes de rentabilidad.

La comparación principal reúne una predicción de cada modelo por ejemplo. Por separado, conservamos todos los años. Si en un año hay una sola clase, no se puede medir su mejora de AUC, aunque sí su diferencia de log-loss. Si falta una predicción necesaria, la comparación correspondiente queda no disponible.

La estabilidad exige una mayoría estricta de los cuarenta años previstos: al menos veintiuno con mejora de AUC. Los años sin AUC estimable permanecen en ese total y no se cuentan como favorables. Se muestra también cuántos pudieron evaluarse. Estos años no son ensayos independientes y algunos contienen pocos ejemplos.
En el gráfico, verde indica mejora y rojo empeoramiento. Una diferencia exactamente cero no produce una barra visible, la cruz gris identifica los años cuyo AUC no puede calcularse.


In [ ]:
from matplotlib.lines import Line2D

columnas_comparacion = ['strategy', 'examples', 'auc_m1', 'auc_m2', 'auc_improvement',
    'log_loss_m1', 'log_loss_m2', 'log_loss_improvement', 'comparison_status']
display(comparacion_regimen.loc[comparacion_regimen.aggregation.eq('pooled'), columnas_comparacion])
display(pd.DataFrame(informe_regimen['stability']))
display(comparacion_regimen.loc[comparacion_regimen.aggregation.eq('year'),
    ['test_year', *columnas_comparacion, 'auc_improved']])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = comparacion_regimen.loc[comparacion_regimen.aggregation.eq('year') & comparacion_regimen.strategy.eq(regla)]
    cambio = datos.auc_improvement.to_numpy(dtype=float)
    ax.bar(datos.test_year.to_numpy(dtype=int), cambio,
           color=np.where(cambio > 0, '#43876b', '#b65b49'), label='Diferencia de AUC')
    sin_auc = datos.loc[datos.auc_improvement.isna(), 'test_year'].to_numpy(dtype=int)
    ax.scatter(sin_auc, np.zeros(len(sin_auc)), marker='x', color='#777777', s=22, label='AUC no estimable')
    ax.axhline(0, color='black', linewidth=.8)
    ax.set(title=regla, ylabel='AUC M2 − AUC M1')
    ax.grid(axis='y', alpha=.2)
axes[0].legend(handles=[Patch(color='#43876b', label='Mejora de AUC'),
    Patch(color='#b65b49', label='Empeoramiento de AUC'),
    Line2D([0], [0], marker='x', color='#777777', linestyle='None', label='AUC no estimable')], fontsize=8)
axes[-1].set_xlabel('Año de prueba')
fig.tight_layout()
plt.show()

## 9. Qué podemos concluir en este punto

Una mejora global permite describir una aportación predictiva provisional del bloque de régimen. Su aceptación requiere todavía las comprobaciones previstas: comparación equivalente en CPCV, sombras específicas, estabilidad y Sharpe neto frente al modelo sin régimen y a las referencias. Tampoco una mejora de clasificación garantiza que las operaciones aceptadas ganen más dinero.

Si M2 empeora o no mejora en suficientes años, se documenta el resultado sin cambiar ventanas, variables o parámetros para favorecerlo. A continuación se estudia M3 con la interacción ya prevista. Las retiradas individuales se muestran en la última parte. La selección interna y la evaluación económica siguen pendientes. El periodo desde 1990 permanece reservado. Siguen vigentes las limitaciones de precios medios mensuales y fuentes reconstruidas.

La siguiente celda guarda cuatro tablas adicionales, vinculadas al intento concreto de M0/M1 que se ha leído. M2 solo añade un candidato por regla, los cuarenta años no son cuarenta configuraciones. Su historial comparte `outputs/experiments/walk_forward_trials.jsonl`, distinguiendo las variantes. Si falla la publicación, repite la celda completa.

In [ ]:
guardados_regimen = []
for nombre, tabla in tablas_regimen.items():
    salida = save_result(ROOT, nombre, tabla, informe_regimen)
    guardados_regimen.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_regimen))
print('Historial:', informe_regimen['trial_log'])
print('Validación de M2 pendiente:', informe_regimen['skill_gate_status'])

## 10. M3: ¿la volatilidad actúa igual después de una caída?

M2 permite que volatilidad y estado del mercado aporten efectos separados. M3 añade la interacción previamente definida: **indicador(retorno de 24 meses < 0) × volatilidad de 12 meses**. Así el modelo puede representar una relación con la volatilidad distinta en un entorno bajista.

La calculamos con los valores originales, antes de normalizarlos. Si el retorno previo de dos años es −10 % y la volatilidad anualizada es 20 %, el producto es 0,20. Si el retorno es positivo o exactamente cero, el producto vale cero. No utilizamos el signo del z-score del retorno: estar por debajo de su media histórica no significa necesariamente haber caído.

Después aplicamos la transformación causal ya usada en el proyecto: recorte a percentiles 1/99 y z-score con al menos 36 meses anteriores. Se calcula sobre el calendario completo, antes de seleccionar las señales de cada regla. La misma interacción acompaña a ambas reglas.

Al principio no hay ningún mercado bajista según este indicador y todo su pasado vale cero. Los límites de recorte son entonces cero y el valor recortado también. Usamos la convención de columna constante: contribución normalizada cero, sin dividir por cero ni inventar una desviación. Esto incluye la primera observación positiva si el recorte causal todavía la lleva a cero. El calentamiento mínimo sigue vigente y los datos ausentes no se rellenan. La misma convención se aplica si los percentiles causales vuelven a recortar todo el pasado a cero. Cuando hay variación histórica después del recorte, se utiliza el z-score habitual. Esta convención se fija antes de ajustar M3 y mantiene la misma muestra de entrenamiento que M1/M2.

In [ ]:
from tfm.data import read_result
from tfm.walk_forward import (INTERACTION_COLUMN, M3_COLUMNS,
    build_interaction_feature, run_walk_forward_interaction)

caracteristicas, _ = read_result(ROOT, 'monthly_features')
interaccion = build_interaction_feature(caracteristicas)
print('Predictores de M3:', M3_COLUMNS)
display(interaccion.loc[interaccion.target_period.between('1930-08', '1931-02')])
print('Meses con convención de historia cero:', int(interaccion.scaling_status.eq('constant_zero_encoded').sum()))

tablas_interaccion, informe_interaccion = run_walk_forward_interaction(ROOT, progress=mostrar_progreso)
comparacion_interaccion = tablas_interaccion['walk_forward_interaction_comparison']
print('Estado del recorrido M3:', informe_interaccion['experiment_status'])

## 11. Separar la aportación de la interacción de la del bloque completo

**M3 frente a M2** mide qué cambia al añadir la interacción. **M3 frente a M1** compara el conjunto completo con la referencia sin régimen explícito. Son dos lecturas del mismo candidato M3, no dos entrenamientos diferentes ni una oportunidad para escoger la comparación más favorable.

Se mantienen los mismos ejemplos, años, parámetros y criterio de estabilidad. La diferencia de AUC es M3 menos la referencia indicada, la mejora de log-loss es la referencia menos M3. Una cifra positiva es favorable. Los empates y años sin AUC estimable se conservan, sin contarlos como mejoras.

El gráfico muestra la comparación más directa, M3 frente a M2. La tabla muestra también M3 frente a M1. Ninguna comparación sustituye a las sombras específicas, la robustez CPCV o la evaluación económica.

In [ ]:
columnas_interaccion = ['reference_model', 'strategy', 'examples', 'auc_reference', 'auc_m3',
    'auc_improvement', 'log_loss_reference', 'log_loss_m3', 'log_loss_improvement', 'comparison_status']
display(comparacion_interaccion.loc[comparacion_interaccion.aggregation.eq('pooled'), columnas_interaccion])
display(pd.DataFrame(informe_interaccion['stability']))
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = comparacion_interaccion.loc[comparacion_interaccion.aggregation.eq('year') &
        comparacion_interaccion.reference_model.eq('M2') & comparacion_interaccion.strategy.eq(regla)]
    cambio = datos.auc_improvement.to_numpy(dtype=float)
    ax.bar(datos.test_year.to_numpy(dtype=int), cambio, color=np.where(cambio > 0, '#43876b', '#b65b49'))
    sin_auc = datos.loc[datos.auc_improvement.isna(), 'test_year'].to_numpy(dtype=int)
    ax.scatter(sin_auc, np.zeros(len(sin_auc)), marker='x', color='#777777', s=22)
    ax.axhline(0, color='black', linewidth=.8)
    ax.set(title=regla, ylabel='AUC M3 − AUC M2')
    ax.grid(axis='y', alpha=.2)
axes[0].legend(handles=[Patch(color='#43876b', label='Mejora de AUC'),
    Patch(color='#b65b49', label='Empeoramiento de AUC'),
    Line2D([0], [0], marker='x', color='#777777', linestyle='None', label='AUC no estimable')], fontsize=8)
axes[-1].set_xlabel('Año de prueba')
fig.tight_layout()
plt.show()

## 12. Guardar M3 y conservar las referencias

La interacción es un candidato previsto antes de ver M2, no se ha elegido probando distintas fórmulas tras sus resultados. Su tratamiento de la historia constante se ha fijado al comprobar cobertura y antes de ajustar M3. Las diez características originales y los cálculos de M0/M1/M2 se conservan.

La ejecución de esta sección lee las referencias publicadas y ajusta únicamente M3. Guarda cuatro tablas con sus JSON y registra una configuración adicional por regla en el historial compartido. Republicar M0/M1 o M2 retira estas salidas para evitar mezclar resultados de intentos distintos. Si falla el guardado, repite la celda completa.

Incluso si mejora, M3 no queda aceptado por estos resultados solamente: siguen pendientes sus contrastes específicos y la simulación económica. No se retocarán la interacción, sus ventanas o el criterio de estabilidad según la comparación obtenida. Antes del commit, limpia las salidas y guarda.

In [ ]:
guardados_interaccion = []
for nombre, tabla in tablas_interaccion.items():
    salida = save_result(ROOT, nombre, tabla, informe_interaccion)
    guardados_interaccion.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_interaccion))
print('Historial:', informe_interaccion['trial_log'])
print('Validación de M3 pendiente:', informe_interaccion['skill_gate_status'])

## 13. Retirar una variable cada vez: qué aporta dentro de M2

M2 reúne volatilidad, retorno de 24 meses y drawdown junto a las dos características de las señales. Ahora quitamos una de esas tres variables de régimen cada vez y **volvemos a entrenar** la logística con las cuatro columnas restantes. Conservamos años, etiquetas, ejemplos, purga y parámetros.

Es una comparación condicionada al resto de entradas. Una variable puede parecer prescindible porque otra recoge información similar. Su resultado tampoco demuestra causalidad ni una importancia válida para cualquier modelo. Este procedimiento se suele llamar *ablación* o retirada individual. Se diferencia de Clustered MDA: allí se desordena un grupo y se utiliza el modelo ya ajustado, aquí se elimina una columna y se ajusta otra vez.

Las tres variantes estaban previstas en el protocolo. No añadimos retiradas de combinaciones, otras ventanas o la interacción de M3. Se calcula un modelo por año, regla y retirada: 240 ajustes adicionales. M2 se lee como referencia, esta sección no lo reentrena ni cambia sus tablas.

In [ ]:
from tfm.walk_forward_ablation import REMOVAL_MODELS, run_walk_forward_ablation

nombres_retiradas = {'z_price_volatility_12m': 'Volatilidad',
    'z_market_return_24m': 'Retorno de 24 meses', 'z_drawdown_12m': 'Drawdown'}
display(pd.DataFrame([{'Variable retirada': nombres_retiradas[c], 'Modelo': m}
                      for c, m in REMOVAL_MODELS.items()]))

def progreso_retiradas(completadas, total):
    if completadas == 1 or completadas % 10 == 0:
        print(f'Bloques de variante y año evaluados: {completadas}/{total}', flush=True)

tablas_retiradas, informe_retiradas = run_walk_forward_ablation(ROOT, progress=progreso_retiradas)
comparacion_retiradas = tablas_retiradas['walk_forward_ablation_comparison']
print('Estado del recorrido:', informe_retiradas['experiment_status'])

## 14. Leer la contribución de conservar la variable

Usamos una orientación distinta a la de «modelo nuevo menos referencia», para describir la aportación de la variable retirada:

- **Contribución de AUC = AUC de M2 completo − AUC sin la variable.**
- **Contribución de log-loss = log-loss sin la variable − log-loss de M2 completo.**

En ambos casos una cifra positiva favorece conservarla. Por ejemplo, si M2 tiene AUC 0,60 y al quitar la variable baja a 0,58, la contribución es +0,02. Si al quitarla sube a 0,62, es −0,02. Un empate en AUC puede acompañarse de probabilidades y log-loss diferentes.

Cada retirada usa los mismos meses. Los años sin AUC estimable siguen presentes. Los recuentos muestran contribuciones positivas, negativas, empates y años no estimables, una mayoría descriptiva tampoco acredita aceptación del régimen.
El AUC reunido también compara ejemplos de años distintos y no es una media de las AUC anuales. Por eso la contribución global puede ser negativa aunque haya más años con contribución positiva que negativa.


In [ ]:
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

principal_retiradas = comparacion_retiradas.loc[comparacion_retiradas.aggregation.eq('pooled')].copy()
principal_retiradas['Variable'] = principal_retiradas.removed_feature.map(nombres_retiradas)
display(principal_retiradas[['strategy', 'Variable', 'examples', 'auc_m2', 'auc_without',
    'auc_contribution', 'log_loss_m2', 'log_loss_without', 'log_loss_contribution', 'comparison_status']])
display(pd.DataFrame(informe_retiradas['contributions']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = principal_retiradas.loc[principal_retiradas.strategy.eq(regla)]
    contribucion = datos.auc_contribution.to_numpy(dtype=float)
    ax.barh(datos.Variable, contribucion, color=np.where(contribucion > 0, '#43876b', '#b65b49'))
    ax.axvline(0, color='black', linewidth=.8)
    ax.set(title=regla, xlabel='AUC de M2 − AUC sin la variable')
    ax.grid(axis='x', alpha=.2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
fig.suptitle('Contribución condicionada a las demás entradas de M2')
fig.tight_layout()
plt.show()

## 15. Un diagnóstico, sin elegir automáticamente las variables

El análisis permite describir qué cambia al retirar cada columna. No convierte la mejor retirada observada en un modelo validado de forma independiente sobre estos mismos años. Si más adelante elegimos un subconjunto o una configuración por sus resultados, esa elección deberá realizarse dentro del entrenamiento y evaluarse por separado.

Se registran las tres variantes, incluidos resultados desfavorables. Los mismos meses aparecen una vez por variante: esas repeticiones no aumentan el número de observaciones independientes. El historial registra cada retirada antes de ajustarla, si falla una, se detiene el recorrido y no se publica una familia parcial. No se modifican M0/M1/M2/M3.

La siguiente celda guarda cuatro tablas `walk_forward_ablation_*`, con la columna retirada y el modelo identificados en cada fila. Republicar M0/M1 o M2 retira estas salidas, M3 no es una entrada y republicarlo las conserva. Si falla el guardado, repite la celda completa. Siguen pendientes la robustez CPCV equivalente, sombras específicas, interpretación y evaluación económica.

In [ ]:
guardados_retiradas = []
for nombre, tabla in tablas_retiradas.items():
    salida = save_result(ROOT, nombre, tabla, informe_retiradas)
    guardados_retiradas.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_retiradas))
print('Historial:', informe_retiradas['trial_log'])
print('Alcance:', informe_retiradas['skill_gate_status'])